# Predicting Student Test Scores 
## Score: 8.70305

In [1]:
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression

In [2]:
train = pd.read_csv('playground-series-s6e1/train.csv')
test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')


In [3]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.015,
    'n_estimators': 8000,
    'num_leaves': 89,
    'max_depth': 11,
    'min_child_samples': 46,
    'reg_alpha': 9.4,
    'reg_lambda': 0.34,
    'min_split_gain': 1e-6,
    'subsample': 0.70,
    'subsample_freq': 3,
    'colsample_bytree': 0.62,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds_gbdt = [420, 666, 80085]
seeds_dart = [420]
n_splits = 10

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

def cv_run(params, seeds, label):
    all_oof = np.zeros(len(X), dtype=float)
    all_test = np.zeros(len(X_test), dtype=float)

    for s_i, seed in enumerate(seeds, start=1):
        p = {**params, 'random_state': seed}

        oof = np.zeros(len(X), dtype=float)
        test_pred = np.zeros(len(X_test), dtype=float)
        rmse_scores = []

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model = lgb.LGBMRegressor(**p)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[lgb.early_stopping(300), lgb.log_evaluation(0)]
            )

            va_pred = model.predict(X_va)
            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f}')

            test_pred += model.predict(X_test) / n_splits

        oof = np.clip(oof, 0, 100)
        test_pred = np.clip(test_pred, 0, 100)

        oof_rmse = float(np.sqrt(mean_squared_error(y, oof)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        all_oof += oof / len(seeds)
        all_test += test_pred / len(seeds)

    all_oof = np.clip(all_oof, 0, 100)
    all_test = np.clip(all_test, 0, 100)

    final_oof_rmse = float(np.sqrt(mean_squared_error(y, all_oof)))
    print(f'{label} FINAL (avg seeds) OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


gbdt_oof, gbdt_test, _ = cv_run(base_params, seeds_gbdt, 'GBDT')

dart_params = {
    **base_params,
    'boosting_type': 'dart',
    'drop_rate': 0.10,
    'skip_drop': 0.50,
    'max_drop': 50
}

dart_oof, dart_test, _ = cv_run(dart_params, seeds_dart, 'DART')

BLEND_W = 0.75
blend_oof = np.clip(BLEND_W * gbdt_oof + (1.0 - BLEND_W) * dart_oof, 0, 100)
blend_test = np.clip(BLEND_W * gbdt_test + (1.0 - BLEND_W) * dart_test, 0, 100)

blend_rmse = float(np.sqrt(mean_squared_error(y, blend_oof)))
print(f'BLEND OOF RMSE: {blend_rmse:.5f}')

y100 = (y >= 99.999).astype(int)
feat_cols = [c for c in ['study_hours', 'class_attendance', 'sleep_hours'] if c in X.columns]

X_clf_train = np.column_stack([blend_oof, blend_oof ** 2] + [X[c].to_numpy(dtype=float) for c in feat_cols])
X_clf_test = np.column_stack([blend_test, blend_test ** 2] + [X_test[c].to_numpy(dtype=float) for c in feat_cols])

final_test = blend_test

if int(y100.sum()) >= 20 and int(y100.sum()) <= (len(y100) - 20):
    clf = LogisticRegression(max_iter=500, class_weight='balanced')
    clf.fit(X_clf_train, y100)

    p100_train = clf.predict_proba(X_clf_train)[:, 1]
    p100_test = clf.predict_proba(X_clf_test)[:, 1]

    CEILING_STRENGTH = 0.15
    P100_THRESHOLD = 0.90
    NEAR_CEILING = 97.0

    adj_oof = blend_oof.copy()
    adj_test = blend_test.copy()

    m_tr = (blend_oof >= NEAR_CEILING) & (p100_train >= P100_THRESHOLD)
    m_te = (blend_test >= NEAR_CEILING) & (p100_test >= P100_THRESHOLD)

    adj_oof[m_tr] = blend_oof[m_tr] + CEILING_STRENGTH * p100_train[m_tr] * (100.0 - blend_oof[m_tr])
    adj_test[m_te] = blend_test[m_te] + CEILING_STRENGTH * p100_test[m_te] * (100.0 - blend_test[m_te])

    adj_oof = np.clip(adj_oof, 0, 100)
    adj_test = np.clip(adj_test, 0, 100)

    adj_rmse = float(np.sqrt(mean_squared_error(y, adj_oof)))
    print(f'CEILING OOF RMSE: {adj_rmse:.5f} | nudged train: {int(m_tr.sum())} test: {int(m_te.sum())}')

    final_test = adj_test
else:
    print('CEILING head skipped')

submission = pd.DataFrame({'id': test_ids, 'exam_score': np.clip(final_test, 0, 100)})
submission.to_csv('submission.csv', index=False)
print('Wrote submission.csv')


GBDT SEED 420 (1/3)
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3447]	valid_0's rmse: 8.70259
  Fold 1/10 RMSE: 8.70259
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3044]	valid_0's rmse: 8.76279
  Fold 2/10 RMSE: 8.76279
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[4103]	valid_0's rmse: 8.70651
  Fold 3/10 RMSE: 8.70651
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3496]	valid_0's rmse: 8.7684
  Fold 4/10 RMSE: 8.76840
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3279]	valid_0's rmse: 8.72533
  Fold 5/10 RMSE: 8.72533
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3321]	valid_0's rmse: 8.74084
  Fold 6/10 RMSE: 8.74084
Training until validation scores don't improve for 300 ro

c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 1/10 RMSE: 8.73249


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 2/10 RMSE: 8.78752


...

  Fold 8/10 RMSE: 8.75467


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 9/10 RMSE: 8.77922


c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\callback.py:333: UserWarning: Early stopping is not available in dart mode
  _log_warning("Early stopping is not available in dart mode")


  Fold 10/10 RMSE: 8.80240
DART Seed 420 OOF RMSE: 8.76676 | Mean fold: 8.76674 (+/- 0.02526)
DART FINAL (avg seeds) OOF RMSE: 8.76676
BLEND OOF RMSE: 8.73593
CEILING OOF RMSE: 8.73602 | nudged train: 3393 test: 1440
Wrote submission.csv
